<a href="https://colab.research.google.com/github/VishwanathSR/House-price-pred/blob/main/House%20price%20prediction%20project/house%20price%20prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
data = pd.read_csv("https://raw.githubusercontent.com/VishwanathSR/House-price-pred/refs/heads/main/House%20price%20prediction%20project/housing.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'housing.csv'

In [ ]:
data

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.dropna(inplace=True)
data.isnull().sum()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x = data.drop(['median_house_value'], axis=1)
y = data['median_house_value']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# EDA
* Now before starting with training the model lets inspect the training data. Use join function to join 2 dataframes x_train & y_train together and analyze them

In [ ]:
train_data = x_train.join(y_train)
train_data

In [ ]:
train_data.hist(figsize=(15, 10))

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(train_data.corr(numeric_only=True), annot=True, cmap="YlGnBu")
# "numeric_onlt=True" is used here to ignore any text based data (e.g. Ocean_proximity) which cannot be plotted.

# Preprocessing (Skewed Curve)
* As we can see in the histogram plot from earlier, there are a few features that are skewed curve, doesnt ressmble gaussian curve. we convert them using log function to get a bell curve that is much more suited to train the model than the raw data.

In [ ]:
train_data['total_rooms'] = np.log(train_data['total_rooms'] + 1)
train_data['total_bedrooms'] = np.log(train_data['total_bedrooms'] + 1)
train_data['population'] = np.log(train_data['population'] + 1)
train_data['households'] = np.log(train_data['households'] + 1)

In [ ]:
train_data.hist(figsize=(15,10))

### (Features with categorical data)
* e.g. Ocean_proximity is a kind of data that cannot be used directly as it is categorical. We can convert each of the categories into multiple features so that we can better train the model.

In [ ]:
train_data.ocean_proximity.value_counts()

In [ ]:
pd.get_dummies(train_data.ocean_proximity)
# pd.get_dummies is used to convert the categorical data into binary(0-1 or True-False), As you can see below one feature is being converted into multiple features.

In [ ]:
# you can add this data into the train_data df using join.
train_data = train_data.join(pd.get_dummies(train_data.ocean_proximity))

In [ ]:
train_data

In [ ]:
# now ocean_proximity is no longer needed, drop it
train_data.drop(["ocean_proximity"], axis=1, inplace=True)
train_data

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(train_data.corr(numeric_only=True), annot=True, cmap="YlGnBu")

In [ ]:
# An interesting vizualization if you have map data(latitude & longitude), hue is what the dot represents in the below example it is the price of house
plt.figure(figsize=(15, 10))
sns.scatterplot(x="latitude", y="longitude", data=train_data, hue="median_house_value", palette="coolwarm")

# Feature Engineering

In [ ]:
train_data['bedroom_ratio'] = train_data['total_bedrooms'] / train_data['total_rooms']
train_data['house_rooms'] = train_data['total_rooms'] / train_data['households']

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(train_data.corr(numeric_only=True), annot=True, cmap="YlGnBu")

# Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train, y_train = train_data.drop(['median_house_value'], axis=1), train_data['median_house_value']

reg = LinearRegression()

reg.fit(x_train, y_train)

In [ ]:
test_data = x_test.join(y_test)

test_data['total_rooms'] = np.log(test_data['total_rooms'] + 1)
test_data['total_bedrooms'] = np.log(test_data['total_bedrooms'] + 1)
test_data['population'] = np.log(test_data['population'] + 1)
test_data['households'] = np.log(test_data['households'] + 1)

test_data = test_data.join(pd.get_dummies(test_data.ocean_proximity))
test_data.drop(["ocean_proximity"], axis=1, inplace=True)

test_data['bedroom_ratio'] = test_data['total_bedrooms'] / test_data['total_rooms']
test_data['house_rooms'] = test_data['total_rooms'] / test_data['households']

In [ ]:
x_test, y_test = test_data.drop(['median_house_value'], axis=1), test_data['median_house_value']
x_train_s = scaler.fit_transform(x_train)

In [ ]:
x_test_s = scaler.transform(x_test)

In [ ]:
reg.score(x_test_s, y_test)

# Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor()

forest.fit(x_train_s, y_train)

In [ ]:
forest.score(x_test_s, y_test)

* Using Cross Validation to get better scoring values

In [ ]:
from sklearn.model_selection import GridSearchCV

forest = RandomForestRegressor()

param_grid = {
    "n_estimators": [200],
    "min_samples_split": [4],
    "max_depth": [None, 4, 8]
}

grid_search = GridSearchCV(forest, param_grid, cv=5,
                          scoring="neg_mean_squared_error",
                          return_train_score=True)
grid_search.fit(x_train_s, y_train)

In [45]:
grid_search.best_estimator_

RandomForestRegressor(min_samples_split=4, n_estimators=200)

In [46]:
grid_search.best_estimator_.score(x_test_s, y_test)

0.8115548104679722